# SOTA Championship V4: Track B Domain-Robust Fine-Tuning Pipeline (Dual Tesla T4)
### WACV 2027 UPAR Challenge & Real-World Surveillance (RWS) Benchmark | Target: mADM >= 0.6850
---
This notebook executes the **Track B Domain-Robust Fine-Tuning** on Kaggle Dual Tesla T4 GPUs.

### Key Enhancements from Track B Plan:
1. **Pretrained Checkpoint Initialization**: Loads weights from `checkpoint_best.pt` (Epoch 26, Val mADM 0.7311) while initializing clean AdamW optimizer buffers.
2. **Domain-Robust Surveillance Augmentation**: Heavy `ColorJitter(0.4)`, `RandAugment(2, 9)`, `RandomErasing(0.5)`, `GaussianBlur(p=0.25)`, `RandomGrayscale(0.1)`, and `RandomResizedCrop`.
3. **Label Smoothing Regularization**: $\epsilon = 0.10$ on multi-label attribute targets to prevent overconfident predictions on unseen surveillance cameras.
4. **Surgical Learning Rate Schedule**: Backbone LR $2 \times 10^{-6}$, Head LR $2 \times 10^{-4}$, 2-epoch linear warmup + 16-epoch cosine annealing.
5. **Automated Submission Packaging**: Extracts clean FP16 weights (~583 MB) and packages `submission_v4_finetune.zip` with Flip TTA and verified air-gapped inference.


In [ ]:
# === CELL 1: HARDWARE INSPECTION & ENVIRONMENT SETUP ===
print("=" * 80)
print("[HARDWARE] Inspecting Dual Tesla T4 Accelerators & Host Environment...")
print("=" * 80)

import torch, os, sys, psutil

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()} | Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU [{i}]: {props.name} | Total VRAM: {props.total_mem / 1e9:.2f} GB")

cpu_count = os.cpu_count() or 4
ram_gb = psutil.virtual_memory().total / 1e9
ram_avail = psutil.virtual_memory().available / 1e9
print(f"Host CPU Cores: {cpu_count} | RAM: {ram_avail:.2f} GB available / {ram_gb:.2f} GB total")
print("=" * 80)


In [ ]:
# === CELL 2: INSTALL DEPENDENCIES ===
import subprocess, sys

print("[DEPS] Installing required packages (open_clip_torch, timm, accelerate, h5py)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "open_clip_torch>=2.24.0", "timm>=1.0.0", "accelerate>=0.33.0", "h5py", "psutil"
])
print("[DEPS] Dependencies installed successfully!")


In [ ]:
# === CELL 3: SETUP WORKING DIRECTORY & REPOSITORY ===
import os, sys, subprocess, shutil, glob

WORK_DIR = "/kaggle/working/upar_challenge_26_rws27"

# 1. First, check if code repository was attached as a Kaggle input dataset
code_candidates = [
    "/kaggle/input/upar-challenge-26-rws27",
    "/kaggle/input/rws2027",
    "/kaggle/input/*rws*",
    "/kaggle/input/*upar*"
]

src_code = None
for pat in code_candidates:
    matches = glob.glob(pat)
    for m in matches:
        if os.path.isdir(m) and os.path.exists(os.path.join(m, "sota_pipeline", "train_v4.py")):
            src_code = m
            break
    if src_code:
        break

if src_code and not os.path.exists(WORK_DIR):
    print(f"[SETUP] Copying codebase from {src_code} -> {WORK_DIR}...")
    shutil.copytree(src_code, WORK_DIR, ignore=shutil.ignore_patterns("*.pt", "*.h5", "__pycache__", ".git"))
    print("[SETUP] Codebase copied successfully!")
elif not os.path.exists(WORK_DIR):
    print("[SETUP] Code not found in /kaggle/input. Cloning directly from GitHub...")
    subprocess.check_call([
        "git", "clone", "https://github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git", WORK_DIR
    ])
    print("[SETUP] Successfully cloned repository from GitHub!")
else:
    print(f"[SETUP] Using existing codebase at {WORK_DIR}")
    if os.path.isdir(os.path.join(WORK_DIR, ".git")):
        try:
            subprocess.run(["git", "-C", WORK_DIR, "pull"], timeout=15)
        except Exception:
            pass

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f"[SETUP] Working directory set to: {os.getcwd()}")


In [ ]:
# === CELL 4: DETECT DATASET CONTAINER & PRETRAINED CHECKPOINT ===
import os, glob

# 1. Detect HDF5 Dataset
h5_candidates = [
    "/kaggle/input/datasets/hinewk/upar-binary-dataset/upar_binary_dataset.h5",
    "/kaggle/input/upar-binary-dataset/upar_binary_dataset.h5",
    "/kaggle/input/*upar*binary*/*.h5",
    os.path.join(WORK_DIR, "upar_binary_dataset.h5"),
]
H5_PATH = None
for pat in h5_candidates:
    matches = glob.glob(pat)
    if matches:
        H5_PATH = matches[0]
        break
if H5_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".h5"):
                H5_PATH = os.path.join(root, f)
                break
        if H5_PATH:
            break

assert H5_PATH and os.path.isfile(H5_PATH), f"HDF5 dataset container not found! Candidates checked: {h5_candidates}"
print(f"[DETECT] HDF5 Dataset: {H5_PATH} ({os.path.getsize(H5_PATH) / 1e6:.1f} MB)")

# 2. Detect Pretrained Checkpoint (Epoch 26 checkpoint_best.pt)
ckpt_candidates = [
    "/kaggle/input/*checkpoint*best*.pt",
    "/kaggle/input/*best*model*.pt",
    "/kaggle/input/*v4*outputs*/*.pt",
    "/kaggle/input/*/*.pt",
    "/kaggle/working/checkpoint_best.pt",
    os.path.join(WORK_DIR, "checkpoint_best.pt"),
]
CHECKPOINT_PATH = None
for pat in ckpt_candidates:
    matches = glob.glob(pat)
    for m in matches:
        fname = os.path.basename(m).lower()
        if "best" in fname or "checkpoint" in fname or "epoch" in fname:
            CHECKPOINT_PATH = m
            break
    if CHECKPOINT_PATH:
        break

if CHECKPOINT_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".pt") and os.path.getsize(os.path.join(root, f)) > 5e8:
                CHECKPOINT_PATH = os.path.join(root, f)
                break
        if CHECKPOINT_PATH:
            break

assert CHECKPOINT_PATH and os.path.isfile(CHECKPOINT_PATH), (
    "Pretrained checkpoint (checkpoint_best.pt) not found in /kaggle/input!\n"
    "Please attach your checkpoint_best.pt as a Kaggle dataset to this notebook."
)
print(f"[DETECT] Pretrained Checkpoint: {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB)")


In [ ]:
# === CELL 5: RAM HYGIENE & GPU CACHE FLUSH ===
import gc, ctypes, torch, psutil

print("[CLEANUP] Performing system RAM garbage collection and malloc_trim...")
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

mem = psutil.virtual_memory()
print(f"[CLEANUP] Available RAM: {mem.available / 1e9:.2f} GB / {mem.total / 1e9:.2f} GB total")
print("[CLEANUP] Ready for high-throughput distributed fine-tuning!")


In [ ]:
# === CELL 6: LAUNCH TRACK B DOMAIN-ROBUST FINE-TUNING ===
import subprocess, sys, os, torch

OUTPUT_DIR = "/kaggle/working/v4_finetune_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters (Track B Recipe)
EPOCHS = 18           # 18 fine-tuning epochs (~10 hours on dual T4)
BATCH_SIZE = 24       # Per-GPU batch size (tuned for 14.0 GB VRAM saturation)
GRAD_ACCUM = 3        # Gradient accumulation steps (24 x 3 x 2 = 144 effective batch size)
BACKBONE_LR = 2e-6    # Surgical fine-tune learning rate for ViT-L/14 backbone (40% of initial)
HEAD_LR = 2e-4        # Surgical fine-tune learning rate for heads (40% of initial)
WARMUP_EPOCHS = 2     # 2-epoch linear warmup
ETA_MIN = 2e-8        # Final cosine decay floor
LABEL_SMOOTHING = 0.1 # Soft label regularization for unseen camera invariance
SAVE_EVERY = 1        # Checkpoint every epoch
EVAL_EVERY = 1        # Validation evaluation every epoch

cmd = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--multi_gpu",
    "--num_processes", str(max(1, torch.cuda.device_count())),
    "--mixed_precision", "fp16",
    os.path.join(WORK_DIR, "sota_pipeline", "train_v4.py"),
    "--h5_path", H5_PATH,
    "--output_dir", OUTPUT_DIR,
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--grad_accum", str(GRAD_ACCUM),
    "--backbone_lr", str(BACKBONE_LR),
    "--head_lr", str(HEAD_LR),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--eta_min", str(ETA_MIN),
    "--label_smoothing", str(LABEL_SMOOTHING),
    "--save_every", str(SAVE_EVERY),
    "--eval_every", str(EVAL_EVERY),
    "--finetune_from", CHECKPOINT_PATH,
    "--strong_aug",
]

print("=" * 80)
print("[TRAIN] Launching Track B Fine-Tuning Engine...")
print(f"[TRAIN] Pretrained Checkpoint: {CHECKPOINT_PATH}")
print(f"[TRAIN] Output Directory: {OUTPUT_DIR}")
print(f"[TRAIN] Effective Batch Size: {BATCH_SIZE} x {GRAD_ACCUM} x {torch.cuda.device_count()} = {BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())}")
print(f"[TRAIN] Learning Rates: Backbone={BACKBONE_LR} | Heads={HEAD_LR}")
print(f"[TRAIN] Schedule: {WARMUP_EPOCHS} warmup epochs + {EPOCHS - WARMUP_EPOCHS} cosine decay epochs")
print(f"[TRAIN] Command:\n{' '.join(cmd)}")
print("=" * 80)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=WORK_DIR,
)

for line in iter(process.stdout.readline, ""):
    print(line, end="", flush=True)

process.wait()
print(f"\n[TRAIN] Fine-tuning finished with exit code: {process.returncode}")
assert process.returncode == 0, f"Training exited with non-zero error code {process.returncode}"


In [ ]:
# === CELL 7: LIST SAVED CHECKPOINTS ===
import os, glob

OUTPUT_DIR = "/kaggle/working/v4_finetune_outputs"
print(f"\n[CHECKPOINTS] Contents of {OUTPUT_DIR}:")
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.pt"))):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  -> {os.path.basename(f)}: {size_mb:.1f} MB")


In [ ]:
# === CELL 8: EXPORT CLEAN FP16 MODEL & RUN AIR-GAPPED VERIFICATION ===
import subprocess, sys, os, shutil, glob

OUTPUT_DIR = "/kaggle/working/v4_finetune_outputs"
SUBMISSION_DIR = os.path.join(WORK_DIR, "submissions")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# 1. Identify best checkpoint
best_ckpt = os.path.join(OUTPUT_DIR, "checkpoint_best.pt")
if not os.path.isfile(best_ckpt):
    ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint_epoch_*.pt")))
    if ckpts:
        best_ckpt = ckpts[-1]
    else:
        best_ckpt = os.path.join(OUTPUT_DIR, "checkpoint_latest.pt")

print(f"[EXPORT] Packaging submission from: {best_ckpt}")

# 2. Run export script to strip optimizer states and cast to FP16
subprocess.check_call([
    sys.executable, os.path.join(WORK_DIR, "sota_pipeline", "export_submission_v4.py"),
    "--checkpoint", best_ckpt,
    "--template_dir", os.path.join(WORK_DIR, "sota_pipeline", "submission_template_v4"),
    "--output_dir", SUBMISSION_DIR,
    "--submission_name", "submission_v4_finetune",
])

# 3. Copy final submission ZIP to /kaggle/working for 1-click download
built_zip = os.path.join(SUBMISSION_DIR, "submission_v4_finetune.zip")
assert os.path.isfile(built_zip), f"Expected submission zip not found at: {built_zip}"

dst_zip = os.path.join("/kaggle/working", "submission_v4_finetune.zip")
shutil.copy2(built_zip, dst_zip)
zip_mb = os.path.getsize(dst_zip) / (1024 * 1024)
print(f"[EXPORT] Final Submission ZIP ready: {dst_zip} ({zip_mb:.2f} MB)")


In [ ]:
# === CELL 9: FINAL VERIFICATION & SUBMISSION INSTRUCTIONS ===
import zipfile, os

ZIP_PATH = "/kaggle/working/submission_v4_finetune.zip"
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print("=" * 80)
    print("[VERIFICATION] Submission ZIP Structure Audit:")
    print(f"  Total archive files: {len(names)}")
    print(f"  run.py present: {'run.py' in names}")
    print(f"  metadata.yaml present: {'metadata.yaml' in names}")
    print(f"  model_v4_fp16.pt present: {'assets/model_v4_fp16.pt' in names}")
    print(f"  vendored timm present: {any('vendor/timm' in n for n in names)}")
    print("=" * 80)
    assert 'run.py' in names and 'metadata.yaml' in names and 'assets/model_v4_fp16.pt' in names

print("\n" + "*" * 80)
print("[DONE] SOTA V4 TRACK B FINE-TUNING COMPLETE!")
print(f"Download: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")
print("Next Steps:")
print("1. Download 'submission_v4_finetune.zip' from the right-hand Output panel in Kaggle.")
print("2. Upload to CodaBench: https://www.codabench.org/competitions/18196/")
print("3. Ingestion & scoring runs automatically on NVIDIA A10 in ~215 seconds.")
print("*" * 80)
